# Building an MDMC Universe
An MDMC simulation requires a configuration and a topology defined within a `Universe` object. These are orthorhombic (cubic or cuboid) boxes containing a configuration of atoms and molecules. This how-to guide will explain how to create a `Universe` object and an atomic configuration inside it.

In addition to creating molecules manually, it is also possible to create them from atomic configuration files (e.g CIF files).  Please see the guide on [Reading atoms from configuration files](read-configurations.ipynb) for a detailed description on how to do this.

In [ ]:
from MDMC.MD import Universe

Creating the actual box that contains the atoms is trivial; specify a list or tuple of 3 floats, representing the x, y, and z lengths of the box respectively; these lengths are taken to be in angstroms (Å).

In [ ]:
box_universe = Universe((10.0, 15.0, 20.0))

To create a cubic `Universe`, only a single float needs to be specified for the `dimensions`; MDMC will infer that this is a cube of side length 10Å. This cube will be used as our base universe for the rest of the guide.

In [ ]:
universe = Universe(10.0)

## Create an atomic configuration
Configurations can either be specified by the user or read from a CIF file.

Each atom is specified using an `Atom` object. At a minimum these can be specified just from an element symbol.

In [ ]:
from MDMC.MD import Atom

# Create some atoms!
H1 = Atom('H')
O1 = Atom('O', position=(8.0, 8.0, 8.0), atom_type=1)
O2 = Atom(element='O', position=(9.0, 9.0, 9.0), velocity=(0., 0., 0.), charge=None)

`Atom`s have a few parameters here! Let's break down some of the particularly important ones:
- `element`: the first parameter gives the element of the atom. If atom `mass` is not given, it will be determined from an elemental lookup table.
- `position`: the (x,y,z) coordinates of the atom. If not given, defaults to the origin.
- `velocity`: the velocity of the atom along the (x,y,z) axes. Defaults to (0,0,0).
- `charge`: the atomic charge of the atom. Defaults to None (no charge).
- `atom_type`: an ID for the type of atom; if not given, is inferred based on the element and interactions of the atom, such that all atoms with the same element and interactions will have the same atom types. This is used to keep track of atom interactions. 

It's recommended you either define no atom types (and let MDMC do it) or define all atom types, to avoid unexpected behaviour.

 Note that if set, the `velocity` of atoms will be scaled when creating a `Simulation` in order to ensure the temperature is accurate. Otherwise, if the velocities of all `Atom` objects in a `Simulation` are 0, then the velocities of the atoms in the MD engine will be randomly chosen in order to provide an accurate temperature. For more details see [Running a Simulation](running-a-simulation.ipynb).

Atoms can also be created by copying another atom and providing the position of the new atom:

In [ ]:
H2 = H1.copy(position=[1., 1., 1.])

The copied atom will have identical properties to the original attribute, except for a different `position`.  This includes interactions, which will apply to the copied atom in the same way as the original atom e.g. if H1 is bonded to an atom O, H2 will also be bonded to atom O.  An example of this is shown in the section **'Create bonded interactions'**.

## Create bonded interactions
There are three bonded interaction types within MDMC: `Bond`, `BondAngle` and `Dihedral`.  Each `Interaction` must have an `InteractionFunction` which describes the `Interaction`.

### `Bond`
This interaction represents a bond between two atoms. Below this is demonstrated where a `Bond` with a `HarmonicPotential` function is created.

In [ ]:
# Import Bond and HarmonicPotential
from MDMC.MD import Bond
from MDMC.MD import HarmonicPotential
# Create a Bond with a HarmonicPotential
# The first argument in the HarmonicPotential is the equilibrium state and the second is the potential strength
HH_bond = Bond(H1, H2, function=HarmonicPotential(1., 100., interaction_type='bond'))

To see which units are supported:

In [ ]:
from MDMC.common.units import SYSTEM
SYSTEM

### `BondAngle`

A `BondAngle` represents bonds at a rotation around a central atom, and is created in the same manner except it requires a minimum of three atoms. The central atom should be the 2nd atom. For example, a water molecule would have:

In [ ]:
from MDMC.MD import BondAngle

H1 = Atom('H')
H2 = Atom('H', position=[0., 1.63298, 0.])
O = Atom('O', position=[0., 0.81649, 0.57736])
HOH_angle = BondAngle(H1, O, H2)

# The following is equivalent
HOH_angle = BondAngle(H2, O, H1)

Currently `HarmonicPotential` is the only `InteractionFunction` that can be applied to either `Bond` or `BondAngle`.

All `Bond` and `BondAngle` interactions can have [constraints](https://en.wikipedia.org/wiki/Constraint_(computational_chemistry)) imposed on them; this can either be set when creating the `Bond` (or `BondAngle`) or afterwards:

In [ ]:
HH_bond.constrained = True
HOH_angle = BondAngle(H1, O, H2, constrained=True)

For a constraint to be applied during a simulation, the `Universe` must have a `ConstraintAlgorithm`. 

*NB: "Constraining" the ability of the bonds to oscillate during MD in this way should not be confused with constraining the value of an MDMC `Parameter` to a certain numerical range during refinement, as described in [Running a Refinement](../../../tutorials/running-a-refinement.ipynb). It is entirely possible to have a rigid bond but allow the length of that bond to change between refinement steps, or conversely have a bond that is free to oscillate during MD but the equilibrium length is not altered as part of the refinement.*

### `DihedralAngle`

A `DihedralAngle` is also created in the same manner, except it requires four atoms.  A `DihedralAngle` can be proper or improper, as specified by `DihedralAngle.improper`.  By default a `DihedralAngle` is proper.

The atoms in a proper `DihedralAngle` must be specified so that the 2nd and 3rd atoms are the central two atoms:

In [ ]:
from MDMC.MD import DihedralAngle

C1 = Atom('C', position=[5.12033922, 4.63847287, 4.94075943])
N = Atom('N', position=[5.12991894, 3.78609704, 3.79996577])
C2 = Atom('C', position=[4.91462725, 4.08992816, 2.5091264])
O = Atom('O', position=[4.67405373, 5.24130678, 2.1180462])
proper = DihedralAngle(atoms=[C1, N, C2, O])

# For a proper DihedralAngle, the equivalent atom order is:
proper = DihedralAngle(atoms=[O, C2, N, C1])

The atoms in an improper `DihedralAngle` must be specific so that the 1st atom is the central atom.

In [ ]:
N = Atom('N', position=[4.97080909, 2.91075722, 1.57280005])
C1 = Atom('C', position=[5.12033922, 4.63847287, 4.94075943])
C2 = Atom('C', position=[5.12991894, 3.78609704, 3.79996577])
C3 = Atom('C', position=[4.91462725, 4.08992816, 2.5091264 ])
improper = DihedralAngle(atoms=[N, C1, C2, C3], improper=True)

# The following are some of the equivalent permutations
# The only unique atom location is the first location (central atom)
improper = DihedralAngle(atoms=[N, C2, C1, C3], improper=True)
improper = DihedralAngle(atoms=[N, C3, C1, C2], improper=True)
improper = DihedralAngle(atoms=[N, C1, C3, C2], improper=True)

Currently `Periodic` is the only `InteractionFunction` that can be applied to `DihedralAngle` interactions.

### Copying bonded atoms

As mentioned above in the **'Create an atom'** section, if an `Atom` with a `BondedInteraction` is copied, the new atom will also have the same `BondedInteraction` (and be bonded to the same atom or atoms as the original).  For example:

In [ ]:
wH1 = Atom('H', position=(5., 5., 5.))
wO = Atom('O', position=(5., 6.63298, 5.))
wBond = Bond((wH1, wO), function=HarmonicPotential(1., 100., interaction_type='bond'))
wH2 = wH1.copy(position=(0., 0.81649, 0.57736))

Both atoms `wH1` and `wH2` are bonded to `wO`:

In [ ]:
wBond.atoms

## Create a molecule
Bonded atoms can be combined into a `Molecule` object, which keeps track of the atoms in the molecule and the bonds between them.

In [ ]:
from MDMC.MD import Molecule, BondAngle
# Create a H2 molecule
H1.position = [0., 0., 0.]
H2.position = [1., 1., 1.]
H_mol = Molecule(position=[2., 1.5, 1.], atoms=[H1, H2], interactions=[HH_bond])

When a `Molecule` is created, the position of the atoms relative to one another is fixed.  The atoms are then moved so that the position of the molecular centre of mass is what was passed when creating the molecule.  In the example above, the atoms were at `[0., 0., 0.]` and `[1., 1., 1.]` before `H_mol` was created; therefore they will always be separated by `1.0 Ang` in each dimension, no matter where the molecule is moved to.  The molecular centre of mass is set to `[2., 1.5, 1.]`, so the atom positions are changed to `[2., 1.5, 1.]` and `[3., 2.5, 2.]` respectively.
It is also possible to `copy` molecules:

In [ ]:
H_mol2 = H_mol.copy(position=[5., 5., 5.])

When a `Molecule` is copied, each `Atom` is copied, as are all of the bonded interactions between these atoms (and all of the non-bonded interactions).

One method for building molecules is to copy atoms, as the interactions are also copied. For example, to build methane:

In [ ]:
m_C = Atom('C')
# Define the H atom positions relative to a C at [0,0,0]
d = 0.629118
H_pos = [[d, d, d], [-d, -d, d], [d, -d, -d], [-d, d, -d]]
m_H = Atom('H', position=H_pos[0])
# Add a bond between 
CH_bonds = Bond(m_C, m_H, function=HarmonicPotential(1.09, 100., interaction_type='bond'))
# Make three H atom copies (which are therefore all bonded to m_C)
H_atoms = [m_H]
for pos in H_pos[1:]:
    H_atoms.append(m_H.copy(pos))
# The next two lines simply create a list of unique HCH triplets
# e.g. [(m_H1, m_C, m_H2), (m_H1, m_C, m_H3), ..., (m_H3, m_C, m_H4)].
from itertools import combinations
HCH_triplets = [(i[0], m_C, i[1]) for i in combinations(H_atoms, 2)]
# Unpack the list of triplets with * notation i.e. [(...), (...), (...)] becomes (...), (...), (...)
HCH_angles = BondAngle(*HCH_triplets, function=HarmonicPotential(109.5, 10., interaction_type='angle'))
# Create a methane molecule by adding C atom to list of H atoms
methane = Molecule(atoms=[m_C]+H_atoms)

## Add structures to a universe
Now we have created our structure objects, we need to add them to our universe box. There are two methods for adding a structure to a universe; either `add_structure`, which adds an individual structure to the universe, or `fill`, which fills the universe with copies of the structure. To `fill`, either specify a desired density of the structure with `num_density` or a desired number of structures with `num_struc_units`.

In [ ]:
# Add an individual structural unit to the universe
universe.add_structure(H_mol)
# Fill the universe with the structural unit repeated on a cubic lattice with a specific number density
universe.fill(H_mol, num_density = 0.01)

Currently the `fill` command cannot be used in conjunction with `add_structure`. When using it with a cubic universe the density will be isotropic, however the exact number of structural units added by `fill` may be lower than expected as it will always add a cube number. Using `fill` with a non-cubic universe is not recommended as the density may or may not be isotropic depending on the dimensions of the universe and the number of units. A list of all structures in the universe can be viewed with `universe.structure_list` (try it!).

## Create non-bonded interactions
Non-bonded interactions are applied to atoms based on their `atom_type`, rather than to individual atoms.  They must also have a `Universe` specified, so that they know which atoms they apply to.  For example, the following code creates a [Lennard-Jones dispersive interaction](https://en.wikipedia.org/wiki/Lennard-Jones_potential) between atom type 1 (here, oxygen atoms) and atom type 2 (here, hydrogen atoms). Note that the parameters for the interactions (here, epsilon and sigma) are used for the refinement of the simulation.

In [ ]:
from MDMC.MD import Dispersion
from MDMC.MD import LennardJones

LJ_HO = Dispersion(universe=universe, atom_types=[1, 2], function=LennardJones(epsilon=0.65, sigma=3.), cutoff=10.)

The exception to this is `Coulombic` interactions, which can be applied either to a list of atoms or to a list of atom types.  If the Coulombic interaction is applied to a list of atoms, the universe does not need to be specified:

In [ ]:
# Import the Coulombic interaction
from MDMC.MD import Coulombic
# Create a Coulombic interaction with a Coulomb potential and a charge of 0.42
c_H = Coulombic(atoms=[H1, H2], charge=0.42)

As `Coulombic` interactions typically have the same interaction function (i.e. a `Coulomb` function, where the force results in Coulomb's law), `Coulombic` interactions do not need to be specified with an interaction function (although other functions can be provided); to set the function as `Coulomb`, a value for the charge can be passed.  The warning highlights the `Coulomb` interaction function has been automatically set.  This is equivalent to:

In [ ]:
from MDMC.MD import Coulomb
c_H = Coulombic(atoms=[H1, H2], function=Coulomb(0.42))

As with all non-bonded interactions, a Coulombic interaction can also be created by specifying the atom types:

In [ ]:
c_O = Coulombic(universe, atom_types=[1], charge=0.42)

In this case a universe must be provided.

As well as manually creating these interactions, it is also possible to apply a `ForceField` to a `Universe`, in order to set the interactions based on experimental data.  Please see the tutorial [Applying a ForceField](applying-a-forcefield.ipynb) for a detailed description on how to do this.

## Solving long-range interactions
More difficult than determining close-range potential energy between atoms is the long-range contribution (beyond the cutoff distance of the interaction) of non-bonded interactions. To deal with this, the MD engines underlying MDMC are equipped with 'KSpace-solvers'. 

There are several solvers for determining the long range energy contribution for non-bonded interactions, including Ewald, particle-particle particle-mesh (PPPM), and particle-mesh Ewald (PME).  If you would like to calculate the long range contribution to the non-bonded energy during a simulation, a KSpace solver has to be added to the universe. A solver can be specified for either electrostatic or dispersive interactions, or both. This can either be during universe initialisation or afterwards:

In [ ]:
# Import Ewald and PPPM kspace solvers
from MDMC.MD import Ewald, PPPM

# Create an ewald solver
ewald = Ewald(accuracy=1e-5)
# Initialise a universe with an Ewald solver for both electrostatics and dispersive interactions
uni1 = Universe(10., kspace_solver=ewald)

# Initialise a universe and then add a PPPM solver for electrostatic interactions
uni2 = Universe(10.)
pppm = PPPM(accuracy=1e-4)
uni2.electrostatic_solver = pppm

# Initialise a universe with a PPPM solver for dispersive interactions
uni3 = Universe(10., dispersive_solver=PPPM)

Not all kspace solvers are implemented for all MD engines, and they may also require different parameters to be specified - see the MD engine documentation for more information.

## Solving bond constraints
In a similar manner to KSpace solvers, [constraint algorithms](https://en.wikipedia.org/wiki/Constraint_(computational_chemistry)) such as SHAKE or RATTLE can also be passed to a universe to solve constraints imposed on bonded interactions. A constraint algorithm is required if any of the bonded interactions are constrained.

In [ ]:
# Import Shake and Rattle
from MDMC.MD import Shake, Rattle
# Initialise a universe with the Shake algorithm
# The first Shake parameter is the accuracy and the second is the maximum number of iterations used for any constraint calculation
shake = Shake(1e-4, 100)
uni4 = Universe(10., constraint_algorithm=shake)
# Add Rattle after universe initialisation
rattle = Rattle(1e-5, 1000)
uni5 = Universe(10.)
uni5.constraint_algorithm = rattle

Not all constraint algorithms are implemented for all MD engines, and they may also required different parameters to be specified - see the MD engine documentation for more information.

## Example Universe filled with water
Here is an example universe filled with water. We use the SPCE [water model](https://en.wikipedia.org/wiki/Water_model) force field to determine interactions.

Note that all water models have (harmonic) potential strengths defined for their `Bond`s and `BondAngle`s. In order to create a rigid water molecule in accordance with the models, a constraint algorithm should be passed to the `Universe` and `constrained=True` passed to the relevant interactions, as shown below.

In [ ]:
from MDMC.MD import *

universe = Universe(dimensions=21.75, constraint_algorithm=Shake(1e-4, 100), electrostatic_solver=PPPM(accuracy=1e-5))
H1 = Atom('H')
H2 = Atom('H', position=(0., 1.63298, 0.))
O = Atom('O', position=(0., 0.81649, 0.57736))
H_coulombic = Coulombic(atoms=[H1, H2], cutoff=10.)
O_coulombic = Coulombic(atoms=O, cutoff=10.)
water_mol = Molecule(position=(0, 0, 0),
                     velocity=(0, 0, 0),
                     atoms=[H1, H2, O],
                     interactions=[Bond((H1, O), (H2, O), constrained=True),
                                   BondAngle(H1, O, H2, constrained=True)],
                     name='water')
universe.fill(water_mol, num_density=0.03356718472021752)
O_dispersion = Dispersion(universe, [O.atom_type, O.atom_type], cutoff=10., vdw_tail_correction=True)
universe.add_force_field('SPCE')

## Accessing information
After creating a `Universe`, the various properties and attributes of it can be accessed as follows:

### Geometry

In [ ]:
print('The dimension(s) of the Universe are {0}, giving a volume of {1}'
      ''.format(universe.dimensions, universe.volume))

### Forcefield

In [ ]:
print('The current force field applied to the Universe is:\n{}'
      ''.format(universe.force_fields))

### Configuration of atoms and molecules

In [ ]:
print('There are {0} atoms in the Universe\n'
      ''.format(universe.n_atoms))

# For a complete list run `print(universe.atoms)`

In [ ]:
print('There are {0} molecules in the Universe\n'
      ''.format(universe.n_molecules))

# For a complete list run `print(universe.molecules)`

### Configuration of interactions

In [ ]:
print('There are {0} total interactions in the Universe\n'
      ''.format(universe.n_interactions))

# For a complete list run `print(universe.interactions)`

In [ ]:
print('There are {0} bonded interactions in the Universe\n'
      ''.format(universe.n_bonded))

# For a complete list run `print(universe.bonded_interactions)`

In [ ]:
print('There are {0} nonbonded interactions in the Universe\n'
      ''.format(universe.n_nonbonded))

# For a complete list run `print(universe.nonbonded_interactions)`

### Constraints

In [ ]:
print('The current constraint algorithm applied to the Universe is {}'
      ''.format(universe.constraint_algorithm.name))

### Solvers
Note that a `kspace_solver` is mutually excusive with the other solver types.

In [ ]:
print('The kspace_solver is {}'.format(universe.kspace_solver and universe.kspace_solver.name))
print('The electrostatic_solver is {}'.format(universe.electrostatic_solver and universe.electrostatic_solver.name))
print('The dispersive_solver is {}'.format(universe.dispersive_solver and universe.dispersive_solver.name))

### Parameters

In [ ]:
print('The MDMC parameters of the Universe are:\n{}'
      ''.format(universe.parameters))